# Fine-tune the case-note student (QLoRA) - AML pipeline V2

**Not run yet.** Written to be run top to bottom on a Colab **L4** GPU (Runtime > Change runtime type).

What it does: trains a LoRA adapter on `train.jsonl` (chat format: system / user / assistant), evaluates on `validation.jsonl` after each epoch, saves the adapter to Google Drive, then writes the student's notes for the 50 dev alerts and, once, the 200 test alerts.

**Size and cost (estimate, not measured).** Train: 1,322 examples, 9.3M Qwen tokens per epoch, longest example 8,177 tokens (all fit 8,192). Expect about 9-10 h on an L4 for 2 epochs (about 550 tokens/s; range 7-11 h) and about 1 h to write the 250 notes: roughly 20 compute units (15-25). Run once with  first to measure the real speed.

Put these in one Drive folder (`DRIVE_DIR` below), from `data/finetune/` of the repo: `train.jsonl`, `validation.jsonl`, `dev_inputs.jsonl`, `test_inputs.jsonl`, and `src/student_generate.py`.

**Model and licence.** Default `Qwen/Qwen2.5-7B-Instruct`: good at JSON, 128k context (we use 8,192). The cell below prints its licence from the Hugging Face model card; the 7B model is listed as Apache-2.0, but read the card before using the adapter beyond this project. Change `MODEL_NAME` to try another instruct model (the chat template and LoRA target names in the config must fit it).

**Data version "blind1".** The student input has no model score and no push sizes (they differ between detectors and gave the first student a shortcut), and decoding stops at 1,400 new tokens with one retry (repetition penalty 1.1) for an invalid answer. Re-upload all five files: the data and `student_generate.py` both changed.

**Never train on the test period.** `train.jsonl` and `validation.jsonl` come from HI-Small; the test inputs are the 200 LI-Small test alerts and are used only in the last cell.

In [ ]:
# Pinned: the latest releases on PyPI when this was written (not tested by the author).
# If one fails to install, change only that line and note it. Then Runtime > Restart.
!pip -q install transformers==5.24.1 peft==0.22.0 accelerate==1.16.0 bitsandbytes==0.50.2 liger-kernel==0.8.5 huggingface_hub==2.4.0

In [ ]:
# ---- Settings (change here, nowhere else) ----
import os
# Must be set before torch is imported (restart the runtime if you ran torch already):
# lets the allocator grow in pieces, which avoids OOM from fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

USE_SMALLER_MODEL = False   # True = Qwen2.5-3B-Instruct, if the 7B still runs out of memory
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # open instruct model, good JSON, long context
if USE_SMALLER_MODEL:
    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # check its licence in the next cell
DRIVE_DIR = "/content/drive/MyDrive/aml_finetune"   # data in, adapter and notes out
MAX_SEQ_LEN = 8192          # prompt + note; longer examples are skipped, never cut
EPOCHS = 2
LEARNING_RATE = 2e-4        # usual for LoRA; the cosine schedule decays it
WARMUP_RATIO = 0.03        # of all optimizer steps; turned into warmup_steps below
PER_DEVICE_BATCH = 1        # 8k tokens per example already fills the GPU
GRADIENT_ACCUMULATION = 16  # effective batch of 16 examples
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
SEED = 0
INFERENCE_BATCH_SIZE = 4   # notes written at once; use 1 if inference runs out of memory
MAX_STEPS = -1            # -1 = full run; set 3 for a quick smoke test first
SAVE_EVERY_STEPS = 20       # about 1.2 h on an L4; a disconnect resumes from here
RUN_NAME = "blind1"          # new data (no score in the input): never resume an older run
MODEL_SLUG = MODEL_NAME.split("/")[-1] + "_" + RUN_NAME   # keeps runs and models apart
ADAPTER_DIR = DRIVE_DIR + "/adapter_" + MODEL_SLUG          # final LoRA adapter
CHECKPOINT_DIR = DRIVE_DIR + "/checkpoints_" + MODEL_SLUG   # lets a disconnect resume

In [ ]:
import json, os, torch
from google.colab import drive
from huggingface_hub import model_info

drive.mount("/content/drive")
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
info = model_info(MODEL_NAME)
print("Model:", MODEL_NAME, "| licence:", info.card_data.license if info.card_data else "unknown")
for name in ["train.jsonl", "validation.jsonl", "dev_inputs.jsonl", "test_inputs.jsonl", "student_generate.py"]:
    assert os.path.exists(os.path.join(DRIVE_DIR, name)), "missing in Drive: " + name

import inspect
import importlib.metadata as metadata
from transformers import TrainingArguments
print({p: metadata.version(p) for p in ["transformers", "peft", "accelerate", "bitsandbytes", "liger-kernel", "torch"]})
# Fail now, not an hour in: every TrainingArguments name used below must exist.
USED_ARGUMENTS = ["output_dir", "num_train_epochs", "learning_rate", "lr_scheduler_type",
                  "warmup_steps", "seed", "bf16", "per_device_train_batch_size",
                  "per_device_eval_batch_size", "gradient_accumulation_steps",
                  "gradient_checkpointing", "gradient_checkpointing_kwargs", "optim",
                  "eval_strategy", "save_strategy", "save_steps", "max_steps",
                  "save_total_limit", "logging_steps", "report_to", "prediction_loss_only"]
unknown = [a for a in USED_ARGUMENTS if a not in inspect.signature(TrainingArguments).parameters]
assert not unknown, "this transformers version has no TrainingArguments: %s" % unknown

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def read_jsonl(name):
    with open(os.path.join(DRIVE_DIR, name)) as handle:
        return [json.loads(line) for line in handle]

def encode(example):
    # Loss only on the assistant's note: the prompt tokens get label -100.
    system_user, full = example["messages"][:2], example["messages"]
    prompt = tokenizer.apply_chat_template(system_user, tokenize=False, add_generation_prompt=True)
    text = tokenizer.apply_chat_template(full, tokenize=False).rstrip("\n")
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    assert ids[:len(prompt_ids)] == prompt_ids, "prompt is not a prefix of the full chat"
    return {"input_ids": ids, "labels": [-100] * len(prompt_ids) + ids[len(prompt_ids):]}

class Encoded(TorchDataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def build(name):
    rows = [encode(e) for e in read_jsonl(name)]
    kept = [r for r in rows if len(r["input_ids"]) <= MAX_SEQ_LEN]
    lengths = sorted(len(r["input_ids"]) for r in rows)
    print(f"{name}: {len(kept)} of {len(rows)} fit {MAX_SEQ_LEN} tokens "
          f"(p50 {lengths[len(lengths)//2]}, max {lengths[-1]})")
    return Encoded(kept)

train_data, validation_data = build("train.jsonl"), build("validation.jsonl")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Liger's fused linear cross-entropy: the loss is computed from the hidden states in
# chunks, so the 8k x 152k logits (about 4 GB) are never built. It must be applied
# BEFORE the model is loaded. Labels are still -100 on the prompt, so the loss stays
# on the assistant's note only. RMSNorm, RoPE and SwiGLU kernels also save memory.
from liger_kernel.transformers import apply_liger_kernel_to_qwen2
apply_liger_kernel_to_qwen2(
    fused_linear_cross_entropy=True, cross_entropy=False,
    rope=True, rms_norm=True, swiglu=True,
)

quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization, device_map="auto",
    dtype=torch.bfloat16, attn_implementation="sdpa",
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

import math
steps_per_epoch = math.ceil(len(train_data) / (PER_DEVICE_BATCH * GRADIENT_ACCUMULATION))
total_steps = MAX_STEPS if MAX_STEPS > 0 else steps_per_epoch * EPOCHS
warmup_steps = max(1, round(WARMUP_RATIO * total_steps))
print(f"{steps_per_epoch} steps per epoch, {total_steps} in all, {warmup_steps} warmup")

class LigerTrainer(Trainer):
    # In evaluation Liger would build the full logits; skip_logits keeps the fused loss.
    def compute_loss(self, model, inputs, *args, **kwargs):
        if self.use_skip_logits:
            inputs = {**inputs, "skip_logits": True}
        return super().compute_loss(model, inputs, *args, **kwargs)

from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM
LigerTrainer.use_skip_logits = "skip_logits" in inspect.signature(Qwen2ForCausalLM.forward).parameters
if not LigerTrainer.use_skip_logits:
    print("WARNING: this Liger/transformers build has no skip_logits; evaluation may need "
          "more memory. If it runs out, set the validation file to fewer examples.")

arguments = TrainingArguments(
    output_dir=CHECKPOINT_DIR, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine", warmup_steps=warmup_steps, seed=SEED, bf16=True,
    per_device_train_batch_size=PER_DEVICE_BATCH, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, prediction_loss_only=True,
    optim="paged_adamw_8bit", eval_strategy="epoch", save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS, max_steps=MAX_STEPS, save_total_limit=3, logging_steps=10, report_to="none",
)
trainer = LigerTrainer(
    model=model, args=arguments, train_dataset=train_data, eval_dataset=validation_data,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
last = get_last_checkpoint(CHECKPOINT_DIR) if os.path.isdir(CHECKPOINT_DIR) else None
print("Resuming from", last) if last else print("Starting fresh")
trainer.train(resume_from_checkpoint=last)

In [ ]:
trainer.save_model(ADAPTER_DIR)          # the LoRA adapter only (about 150 MB)
tokenizer.save_pretrained(ADAPTER_DIR)
history = [h for h in trainer.state.log_history if "eval_loss" in h]
print("Validation loss by epoch:", [(h["epoch"], round(h["eval_loss"], 4)) for h in history])
print("Saved adapter to", ADAPTER_DIR)
# Rising validation loss in epoch 2 means overfitting: use the epoch-1 checkpoint in
# CHECKPOINT_DIR instead (point ADAPTER_DIR at it).

## Student notes (inference)

The training model is freed first: inference runs in a separate process that loads its own copy, and both do not fit in the GPU. The dev run picks nothing automatically: look at it before the test run. **The test run is meant to happen once.**

In [ ]:
import gc

# A smoke test (MAX_STEPS > 0) stops here, so "Run all" never writes notes from a
# 3-step adapter. Set MAX_STEPS = -1 for the real run.
if MAX_STEPS > 0:
    raise RuntimeError("Smoke test finished OK. Set MAX_STEPS = -1 for the real run.")

# Free the training model, optimizer and cached memory held by this notebook.
del trainer, model
gc.collect()
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Free GPU memory: {free_gb:.1f} GB")
assert free_gb > 16, "GPU memory is still held by this notebook: Runtime > Restart session, "     "then run only the settings cell and the two inference cells (the adapter is on Drive)."

In [ ]:
# 50 dev alerts (LI-Small validation period)
!python {DRIVE_DIR}/student_generate.py --inputs {DRIVE_DIR}/dev_inputs.jsonl --output {DRIVE_DIR}/dev_raw.jsonl --adapter {ADAPTER_DIR} --base-model {MODEL_NAME} --batch-size {INFERENCE_BATCH_SIZE}

In [ ]:
# 200 test alerts: run ONCE, after the dev result looks right. It skips finished alerts.
!python {DRIVE_DIR}/student_generate.py --inputs {DRIVE_DIR}/test_inputs.jsonl --output {DRIVE_DIR}/test_raw.jsonl --adapter {ADAPTER_DIR} --base-model {MODEL_NAME} --batch-size {INFERENCE_BATCH_SIZE}

## Score on your own computer (CPU)

Download `dev_raw.jsonl` and `test_raw.jsonl` into `data/finetune/` of the repo, then:

```bash
python -m src.score_student --set dev  --raw data/finetune/dev_raw.jsonl
python -m src.score_student --set test --raw data/finetune/test_raw.jsonl
```

Same fact check and the same five measures (`src/eval_v2.py`) as V1 and prompt v3.